# reduce-gather-sum — worked example 3: Build a stacked matrix from per-rank row vectors

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reduce-gather-sum`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When each rank owns a row vector and you want them assembled into one matrix on a single rank, `gather` (or `all_gather`) collects the per-rank tensors into a list, and `t.stack` turns that list into a new leading axis. Gather preserves which row came from which rank, unlike a reduction that would collapse them.

## Worked solution

Each rank holds a `(D,)` row. `FakeDist.all_gather` fills a pre-allocated list with every rank's row. The function `stack_rows_via_gather` allocates `gather_list = [zeros(D) for _ in range(world_size)]`, gathers, then `t.stack(gather_list, dim=0)` to produce a `(world_size, D)` matrix where row r is rank r's vector. We print the resulting shape and the matrix. The key idea: `stack` adds a new axis to assemble vectors into a matrix, which is exactly the post-gather assembly step.

In [ ]:
class FakeDist:
    def __init__(self, rows):
        self.rows = [r.clone() for r in rows]
    def all_gather(self, out_list, tensor):
        for i, r in enumerate(self.rows):
            out_list[i].copy_(r)


def stack_rows_via_gather(world_size, D, dist_module):
    gather_list = [t.zeros(D) for _ in range(world_size)]
    dist_module.all_gather(gather_list, t.zeros(D))
    return t.stack(gather_list, dim=0)


rows = [t.tensor([1.0, 2.0]), t.tensor([3.0, 4.0]), t.tensor([5.0, 6.0])]
fd = FakeDist(rows)
mat = stack_rows_via_gather(len(rows), 2, fd)
print('shape:', tuple(mat.shape))
print(mat.tolist())